<a href="https://colab.research.google.com/github/HarithaGottumukkala/DATA266-1598_hw5/blob/main/DATA266_1598_hw5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATA 266 — Fine-Tuning LLM using LoRA (PEFT)

## Assignment Objective

In this assignment, I will fine-tune a pretrained language model on the DialogSum dataset using LoRA (Low-Rank Adaptation).

I will first evaluate the pretrained model before fine-tuning, then train LoRA adapters on the dialogue summarization task, and compare the model outputs before and after training.

I will also compare two LoRA ranks, r = 4 and r = 16, to study how the adapter size affects the number of trainable parameters and the quality of the generated summaries.

## Standing Requirement — Personal Parameters

- **SID4:** 1598
- **SEED:** 1598
- **SLICE:** 598
- **HP_ID:** 2
- **CLS_A:** 8
- **CLS_B:** 5

## 0. Environment Setup and Reproducibility

Before starting the assignment, I set up the required libraries, configure the random seeds, and verify the GPU environment.

Using a fixed seed helps make the experiment more reproducible across repeated runs.

In [1]:
%pip uninstall -y torchao
%pip install -q transformers datasets peft accelerate sentencepiece evaluate rouge_score

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00


### 0.1 Personal Parameters and Random Seeds

I use my assigned SEED = 1598 for Python, NumPy, and PyTorch so that random operations such as sampling and shuffling are as reproducible as possible.

In [2]:
import os
import random
import numpy as np
import torch

# Personal parameters
SID4 = 1598
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

# Reproducibility
os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.use_deterministic_algorithms(True, warn_only=True)

print("Personal Parameters")
print("-" * 40)
print(f"SID4   : {SID4}")
print(f"SEED   : {SEED}")
print(f"SLICE  : {SLICE}")
print(f"HP_ID  : {HP_ID}")
print(f"CLS_A  : {CLS_A}")
print(f"CLS_B  : {CLS_B}")

Personal Parameters
----------------------------------------
SID4   : 1598
SEED   : 1598
SLICE  : 598
HP_ID  : 2
CLS_A  : 8
CLS_B  : 5


### 0.2 Runtime and GPU Verification

I verify that the notebook is running on a GPU before loading the model. This makes LoRA fine-tuning much faster than running the same experiment on a CPU.

In [3]:
print("Runtime Information")
print("-" * 50)

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version     : {torch.version.cuda}")

    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU memory       : {gpu_memory:.2f} GB")
else:
    print("GPU was not detected. Please change the Colab runtime to T4 GPU.")

Runtime Information
--------------------------------------------------
PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU              : Tesla T4
CUDA version     : 12.8
GPU memory       : 14.56 GB


In [4]:
import transformers
import datasets
import peft
import accelerate
import importlib.util

print("Library Versions")
print("-" * 50)
print(f"Transformers : {transformers.__version__}")
print(f"Datasets     : {datasets.__version__}")
print(f"PEFT         : {peft.__version__}")
print(f"Accelerate   : {accelerate.__version__}")
print(f"PyTorch      : {torch.__version__}")

torchao_available = importlib.util.find_spec("torchao") is not None
print(f"torchao available: {torchao_available}")

Library Versions
--------------------------------------------------
Transformers : 5.16.1
Datasets     : 4.8.5
PEFT         : 0.20.0
Accelerate   : 1.14.0
PyTorch      : 2.11.0+cu128
torchao available: False


## 1. Data Preparation


Load the DialogSum dataset and prepare it for dialogue summarization.

The required format is:

- **Input:** dialogue
- **Target:** summary

I will also display two processed samples to verify that the data is prepared correctly.

### Approach

The DialogSum dataset contains conversations along with human-written summaries.

For this task, I use the dialogue as the model input and the corresponding summary as the target output. Before preprocessing the data, I first inspect the dataset structure and available splits.

In [5]:
from datasets import load_dataset

dataset = load_dataset("neil-code/dialogsum-test")

print("DialogSum Dataset")
print("-" * 50)
print(dataset)

README.md:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/1.81M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/441k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/447k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1999 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/499 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/499 [00:00<?, ? examples/s]

DialogSum Dataset
--------------------------------------------------
DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1999
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
})


In [6]:
print("Dataset Information")
print("-" * 50)

for split_name in dataset.keys():
    print(f"{split_name.capitalize():10s}: {len(dataset[split_name]):,} examples")

print("\nColumns:")
print(dataset["train"].column_names)

Dataset Information
--------------------------------------------------
Train     : 1,999 examples
Validation: 499 examples
Test      : 499 examples

Columns:
['id', 'dialogue', 'summary', 'topic']


### 1.1 Inspecting a Raw Example

Before changing the dataset, I inspect one training example to understand how the dialogue and summary are stored.

In [7]:
sample = dataset["train"][0]

print("Raw Training Example")
print("=" * 70)

print("\nID:")
print(sample["id"])

print("\nDIALOGUE:")
print(sample["dialogue"])

print("\nSUMMARY:")
print(sample["summary"])

print("\nTOPIC:")
print(sample["topic"])

Raw Training Example

ID:
train_0

DIALOGUE:
#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.
#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.
#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.
#Person2#: Ok, thanks doctor.

SUMMARY:
Mr. Smith's g

### 1.2 Preparing the Input and Target

For each example:

- The dialogue is used as the **input**.
- The human-written summary is used as the **target**.

I add a short instruction before the dialogue so that FLAN-T5 clearly understands that the task is summarization.

In [8]:
def prepare_example(example):
    input_text = (
        "Summarize the following dialogue:\n\n"
        + example["dialogue"]
    )

    target_text = example["summary"]

    return {
        "input_text": input_text,
        "target_text": target_text
    }


processed_dataset = dataset.map(prepare_example)

print("Preprocessing completed.")
print("\nNew columns:")
print(processed_dataset["train"].column_names)

Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Preprocessing completed.

New columns:
['id', 'dialogue', 'summary', 'topic', 'input_text', 'target_text']


### 1.3 Two Processed Samples

The following two examples show the final input-target format that will later be used for model training.

In [9]:
for i in range(2):
    example = processed_dataset["train"][i]

    print("=" * 80)
    print(f"PROCESSED SAMPLE {i + 1}")
    print("=" * 80)

    print("\nINPUT:")
    print(example["input_text"])

    print("\nTARGET:")
    print(example["target_text"])

    print()

PROCESSED SAMPLE 1

INPUT:
Summarize the following dialogue:

#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.
#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.
#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.
#Person2#: Ok, thanks doctor.

TARGE

## 2. Baseline Inference



Load a pretrained model and generate summaries for two dialogues before fine-tuning.

These outputs will be used as the baseline for comparison with the LoRA fine-tuned model later.

### Approach

I use `google/flan-t5-small` as the pretrained sequence-to-sequence model.

I select two examples from the DialogSum test split using my assigned seed so that the same dialogues can be reproduced and reused after fine-tuning.

In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"

print(f"Loading model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_model = base_model.to(device)

print("\nModel loaded successfully.")
print(f"Device: {device}")

Loading model: google/flan-t5-small


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


Model loaded successfully.
Device: cuda


### 2.1 Selecting Two Test Dialogues

I select two examples from the test split using `SEED = 1598`.

These two examples will remain fixed for the rest of the assignment so that the baseline, `r = 4`, and `r = 16` models are evaluated on exactly the same dialogues.

In [11]:
rng = random.Random(SEED)

eval_indices = rng.sample(
    range(len(processed_dataset["test"])),
    2
)

print("Selected test indices:", eval_indices)

Selected test indices: [240, 216]


In [12]:
eval_examples = [
    processed_dataset["test"][index]
    for index in eval_indices
]

print(f"Saved {len(eval_examples)} evaluation examples.")

Saved 2 evaluation examples.


### 2.2 Evaluation Examples

Before generating the baseline summaries, I display the two selected dialogues together with their reference summaries.

In [13]:
for i, example in enumerate(eval_examples, start=1):

    print("=" * 80)
    print(f"EVALUATION EXAMPLE {i}")
    print("=" * 80)

    print("\nDIALOGUE:")
    print(example["dialogue"])

    print("\nREFERENCE SUMMARY:")
    print(example["summary"])

    print()

EVALUATION EXAMPLE 1

DIALOGUE:
#Person1#: Hello. Is this ABC Rent-a-car Company?
#Person2#: Yes, speaking. May I help you?
#Person1#: This morning we rented a car and we are on the way to Niagara Falls. I'm afraid we have a car accident near the border.
#Person2#: That's too bad. What kind of accident is it? Are you all right?
#Person1#: I'm all right, but my friend is seriously injured. Will you call an ambulance and the police?
#Person2#: OK. I'll do it right away, but tell me how it happened.
#Person1#: I ran into the guardrail when I turned to the left.

REFERENCE SUMMARY:
#Person1# rent a car from ABC Rent-a-car Company this morning and met an accident. #Person2# will call an ambulance and police for #Person1#.

EVALUATION EXAMPLE 2

DIALOGUE:
#Person1#: We need to do a group report tomorrow. I need to go to the stationer to buy something for the posters.
#Person2#: You need to make posters?
#Person1#: absolutely. Every time when we do the reports, we love making some beautiful p

### 2.3 Baseline Generation

I now generate summaries using the original pretrained FLAN-T5 model.

I keep the generation settings fixed so that the same settings can be reused after fine-tuning for a fair comparison.

In [14]:
def generate_summary(model, input_text):
    model.eval()

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            num_beams=4,
            do_sample=False
        )

    summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return summary

In [15]:
baseline_results = []

for i, example in enumerate(eval_examples, start=1):

    baseline_summary = generate_summary(
        base_model,
        example["input_text"]
    )

    baseline_results.append({
        "example": i,
        "test_index": eval_indices[i - 1],
        "dialogue": example["dialogue"],
        "reference_summary": example["summary"],
        "baseline_summary": baseline_summary
    })

print("Baseline inference completed.")

Baseline inference completed.


### 2.4 Baseline Results

The following outputs show how the original pretrained model summarizes the two test dialogues before any LoRA fine-tuning.

In [16]:
for result in baseline_results:

    print("=" * 80)
    print(f"BASELINE EXAMPLE {result['example']}")
    print("=" * 80)

    print("\nREFERENCE SUMMARY:")
    print(result["reference_summary"])

    print("\nBASELINE MODEL SUMMARY:")
    print(result["baseline_summary"])

    print()

BASELINE EXAMPLE 1

REFERENCE SUMMARY:
#Person1# rent a car from ABC Rent-a-car Company this morning and met an accident. #Person2# will call an ambulance and police for #Person1#.

BASELINE MODEL SUMMARY:
I ran into the guardrail and got into a car.

BASELINE EXAMPLE 2

REFERENCE SUMMARY:
#Person1# and #Person2# will do a group report, so they plan to buy something to make posters. They make a shopping list before they go and find items according to signs on shelves.

BASELINE MODEL SUMMARY:
I need to do a group report tomorrow. I need to do a group report tomorrow. I need to go to the stationer to buy something for the posters. I need markers, color pencils, a pair of scissors, a pair of highlighter, rulers. I need markers, color pencils and highlighters. I need markers, color pencils and highlighters



In [17]:
import json

with open("baseline_outputs.json", "w") as file:
    json.dump(
        baseline_results,
        file,
        indent=2
    )

print("Baseline outputs saved to baseline_outputs.json")

Baseline outputs saved to baseline_outputs.json


## 3. Apply LoRA


Apply LoRA using PEFT to the pretrained model.

For the first experiment, I use:

- **LoRA rank (`r`)**: 4
- **LoRA alpha**: 16
- **LoRA dropout**: 0.05
- **Target modules**: query (`q`) and value (`v`) projection layers

After attaching the LoRA adapters, I report the total number of parameters and the number of trainable parameters.

### Approach

LoRA keeps the original pretrained model weights frozen and adds small trainable adapter matrices to selected layers.

This allows the model to learn the dialogue summarization task while updating only a small fraction of the full model parameters.

In [18]:
from peft import LoraConfig, TaskType, get_peft_model

print("PEFT components imported successfully.")

PEFT components imported successfully.


### 3.1 LoRA Configuration

I first define the LoRA settings for the `r = 4` experiment.

The adapter is attached to the query and value projection layers of the attention mechanism, while the original FLAN-T5 weights remain frozen.

In [19]:
LORA_RANK = 4
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

lora_config_r4 = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q", "v"],
    bias="none"
)

print("LoRA Configuration")
print("-" * 50)
print(f"Rank (r)       : {LORA_RANK}")
print(f"Alpha          : {LORA_ALPHA}")
print(f"Dropout        : {LORA_DROPOUT}")
print('Target modules : ["q", "v"]')
print("Task type      : SEQ_2_SEQ_LM")

LoRA Configuration
--------------------------------------------------
Rank (r)       : 4
Alpha          : 16
Dropout        : 0.05
Target modules : ["q", "v"]
Task type      : SEQ_2_SEQ_LM


### 3.2 Loading a Fresh Base Model

I load a fresh copy of FLAN-T5-small before attaching the LoRA adapter.

This keeps the baseline model separate and ensures that the `r = 4` and `r = 16` experiments both start from the same pretrained model.

In [20]:
model_r4 = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model_r4 = model_r4.to(device)

print("Fresh FLAN-T5-small model loaded for the r=4 experiment.")
print(f"Device: {device}")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Fresh FLAN-T5-small model loaded for the r=4 experiment.
Device: cuda


In [21]:
model_r4 = get_peft_model(
    model_r4,
    lora_config_r4
)

print("LoRA adapters attached successfully.")

LoRA adapters attached successfully.


### 3.3 Total and Trainable Parameters

I now compare the total number of parameters with the number of parameters that will actually be updated during training.

This shows how parameter-efficient LoRA is compared with full fine-tuning.

In [22]:
def count_parameters(model):
    total_params = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    trainable_params = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    trainable_percentage = (
        100 * trainable_params / total_params
    )

    return total_params, trainable_params, trainable_percentage


total_params_r4, trainable_params_r4, trainable_percent_r4 = count_parameters(
    model_r4
)

print("Parameter Summary — LoRA r=4")
print("-" * 50)
print(f"Total parameters     : {total_params_r4:,}")
print(f"Trainable parameters : {trainable_params_r4:,}")
print(f"Trainable percentage : {trainable_percent_r4:.4f}%")

Parameter Summary — LoRA r=4
--------------------------------------------------
Total parameters     : 77,133,184
Trainable parameters : 172,032
Trainable percentage : 0.2230%


In [23]:
model_r4.print_trainable_parameters()

trainable params: 172,032 || all params: 77,133,184 || trainable%: 0.2230


### 3.4 Verifying the LoRA Adapter Parameters

I inspect a few trainable parameter names to confirm that the parameters being updated belong to the LoRA adapters.

In [24]:
print("First 10 Trainable Parameters")
print("-" * 70)

count = 0

for name, parameter in model_r4.named_parameters():
    if parameter.requires_grad:
        print(name)
        count += 1

    if count == 10:
        break

First 10 Trainable Parameters
----------------------------------------------------------------------
base_model.model.encoder.block.0.layer.0.SelfAttention.q.lora_A.default.weight
base_model.model.encoder.block.0.layer.0.SelfAttention.q.lora_B.default.weight
base_model.model.encoder.block.0.layer.0.SelfAttention.v.lora_A.default.weight
base_model.model.encoder.block.0.layer.0.SelfAttention.v.lora_B.default.weight
base_model.model.encoder.block.1.layer.0.SelfAttention.q.lora_A.default.weight
base_model.model.encoder.block.1.layer.0.SelfAttention.q.lora_B.default.weight
base_model.model.encoder.block.1.layer.0.SelfAttention.v.lora_A.default.weight
base_model.model.encoder.block.1.layer.0.SelfAttention.v.lora_B.default.weight
base_model.model.encoder.block.2.layer.0.SelfAttention.q.lora_A.default.weight
base_model.model.encoder.block.2.layer.0.SelfAttention.q.lora_B.default.weight


In [25]:
experiment_r4_info = {
    "model": MODEL_NAME,
    "rank": LORA_RANK,
    "alpha": LORA_ALPHA,
    "dropout": LORA_DROPOUT,
    "target_modules": ["q", "v"],
    "total_parameters": total_params_r4,
    "trainable_parameters": trainable_params_r4,
    "trainable_percentage": trainable_percent_r4
}

print("r=4 experiment information saved.")

r=4 experiment information saved.


### Observation

The number of trainable parameters is much smaller than the total number of model parameters. This confirms that LoRA is updating only a small adapter portion of the model instead of fine-tuning all FLAN-T5 parameters.

## 4. Fine-Tuning



Fine-tune the LoRA-enabled model on the DialogSum dataset.

For this experiment, I train the `r = 4` LoRA adapter while keeping the original FLAN-T5 weights frozen. I also report the training logs and validation loss, and save the trained LoRA adapter after training.

### Approach

I use the available DialogSum training and validation splits for fine-tuning.

The text is first tokenized into model-readable input IDs and target labels. I use the same training data and training settings later for the `r = 16` experiment so that the rank comparison remains fair.

In [26]:
train_data = processed_dataset["train"]
validation_data = processed_dataset["validation"]

print("Training Data")
print("-" * 45)
print(f"Training examples   : {len(train_data):,}")
print(f"Validation examples : {len(validation_data):,}")

Training Data
---------------------------------------------
Training examples   : 1,999
Validation examples : 499


### 4.1 Tokenizing the Dataset

The model cannot directly process raw text, so I convert the dialogue inputs and target summaries into token IDs.

I limit the dialogue input to 512 tokens and the target summary to 128 tokens.

In [27]:
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128


def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [28]:
tokenized_train = train_data.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_data.column_names
)

tokenized_validation = validation_data.map(
    tokenize_batch,
    batched=True,
    remove_columns=validation_data.column_names
)

print("Tokenization completed.")

print("\nTraining columns:")
print(tokenized_train.column_names)

print("\nValidation columns:")
print(tokenized_validation.column_names)

Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Tokenization completed.

Training columns:
['input_ids', 'attention_mask', 'labels']

Validation columns:
['input_ids', 'attention_mask', 'labels']


### 4.2 Tokenization Check

Before training, I inspect one tokenized example to confirm that the input and target labels were created correctly.

In [29]:
tokenized_example = tokenized_train[0]

print("Tokenized Example")
print("-" * 50)
print(f"Input token count  : {len(tokenized_example['input_ids'])}")
print(f"Target token count : {len(tokenized_example['labels'])}")

print("\nAvailable fields:")
print(tokenized_example.keys())

print("\nDecoded target:")
print(
    tokenizer.decode(
        tokenized_example["labels"],
        skip_special_tokens=True
    )
)

Tokenized Example
--------------------------------------------------
Input token count  : 291
Target token count : 48

Available fields:
dict_keys(['input_ids', 'attention_mask', 'labels'])

Decoded target:
Mr. Smith's getting a check-up, and Doctor Hawkins advises him to have one every year. Hawkins'll give some information about their classes and medications to help Mr. Smith quit smoking.


### 4.3 Dynamic Padding

The dialogues have different lengths. I use a sequence-to-sequence data collator to pad only the examples inside each batch as needed.

This avoids unnecessary padding and uses GPU memory more efficiently.

In [30]:
from transformers import DataCollatorForSeq2Seq

data_collator_r4 = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_r4
)

print("Data collator created successfully.")

Data collator created successfully.


### 4.4 Training Configuration

I train the `r = 4` LoRA adapter for two epochs.

The same learning rate, batch size, precision, and other training settings will later be reused for the `r = 16` experiment. I use FP32 for stable training on this model.

In [31]:
from transformers import Seq2SeqTrainingArguments

OUTPUT_DIR_R4 = "dialogsum_lora_r4"

training_args_r4 = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR_R4,

    num_train_epochs=2,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    learning_rate=2e-4,
    weight_decay=0.01,
    max_grad_norm=1.0,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=25,

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=False,

    report_to="none",

    seed=SEED,
    data_seed=SEED
)

print("Training configuration created for LoRA r=4.")

Training configuration created for LoRA r=4.


In [32]:
from transformers import Seq2SeqTrainer

trainer_r4 = Seq2SeqTrainer(
    model=model_r4,
    args=training_args_r4,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator_r4,
    processing_class=tokenizer
)

print("Trainer created successfully for LoRA r=4.")

Trainer created successfully for LoRA r=4.


### 4.5 Pre-Training Sanity Check

Before starting the complete training run, I calculate the loss on a small batch.

A finite positive loss confirms that the tokenized inputs, target labels, and model forward pass are working correctly.

In [33]:
test_batch_r4 = data_collator_r4(
    [
        tokenized_train[0],
        tokenized_train[1]
    ]
)

test_batch_r4 = {
    key: value.to(device)
    for key, value in test_batch_r4.items()
}

model_r4.eval()

with torch.no_grad():
    test_output_r4 = model_r4(**test_batch_r4)

pretraining_loss_r4 = test_output_r4.loss.item()

print("Pre-Training Sanity Check")
print("-" * 50)
print(f"Batch loss: {pretraining_loss_r4:.4f}")

Pre-Training Sanity Check
--------------------------------------------------
Batch loss: 2.2771


### 4.6 Fine-Tuning the r = 4 Adapter

The sanity check produced a valid loss, so I now start the complete fine-tuning run.

During training, the original FLAN-T5 parameters remain frozen and only the LoRA adapter parameters are updated.

In [34]:
import time

start_time_r4 = time.time()

train_result_r4 = trainer_r4.train()

training_time_r4 = time.time() - start_time_r4

print("\nTraining completed.")
print(f"Training time: {training_time_r4 / 60:.2f} minutes")

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch,Training Loss,Validation Loss
1,3.352018,1.437019
2,3.206309,1.414560



Training completed.
Training time: 2.70 minutes


### 4.7 Training Results

After fine-tuning, I record the average training loss, validation loss, number of training steps, and total training time.

In [35]:
average_training_loss_r4 = train_result_r4.training_loss

eval_results_r4 = trainer_r4.evaluate()
validation_loss_r4 = eval_results_r4["eval_loss"]

print("LoRA r=4 Training Results")
print("-" * 50)
print(f"Average training loss : {average_training_loss_r4:.4f}")
print(f"Validation loss       : {validation_loss_r4:.4f}")
print(f"Global training steps : {train_result_r4.global_step}")
print(f"Training time         : {training_time_r4 / 60:.2f} minutes")

Training Loss,Validation Loss,Epoch
3.206309,1.414560,2


LoRA r=4 Training Results
--------------------------------------------------
Average training loss : 3.4834
Validation loss       : 1.4146
Global training steps : 250
Training time         : 2.70 minutes


### 4.8 Saving the Fine-Tuned Adapter

After training, I save the LoRA adapter separately from the original FLAN-T5 model.

This keeps the saved fine-tuned component small because only the learned LoRA parameters need to be stored.

In [36]:
R4_ADAPTER_PATH = "dialogsum_lora_r4_adapter"

model_r4.save_pretrained(R4_ADAPTER_PATH)
tokenizer.save_pretrained(R4_ADAPTER_PATH)

print(f"LoRA r=4 adapter saved to: {R4_ADAPTER_PATH}")

LoRA r=4 adapter saved to: dialogsum_lora_r4_adapter


In [37]:
experiment_r4_info.update({
    "training_examples": len(train_data),
    "validation_examples": len(validation_data),
    "epochs": 2,
    "learning_rate": 2e-4,
    "batch_size": 8,
    "gradient_accumulation_steps": 2,
    "precision": "FP32",
    "pretraining_batch_loss": pretraining_loss_r4,
    "average_training_loss": average_training_loss_r4,
    "validation_loss": validation_loss_r4,
    "global_steps": train_result_r4.global_step,
    "training_time_minutes": training_time_r4 / 60
})

print("Final r=4 experiment metrics saved.")

Final r=4 experiment metrics saved.


### Observation

The r = 4 LoRA adapter completed training successfully and produced finite training and validation losses. This confirms that the model was able to learn from the DialogSum examples while training only the LoRA adapter parameters.

## 5. Post-Fine-Tuning Inference



Run inference again on the same two dialogues used for the baseline experiment.

The goal is to observe how the summaries change after LoRA fine-tuning.

### Approach

I reuse the same two test dialogues selected during baseline inference.

I also use the same generation settings so that the before-and-after comparison is fair.

In [38]:
model_r4.eval()

print("Fine-tuned LoRA r=4 model is ready for inference.")

Fine-tuned LoRA r=4 model is ready for inference.


In [39]:
r4_results = []

for i, example in enumerate(eval_examples, start=1):

    fine_tuned_summary = generate_summary(
        model_r4,
        example["input_text"]
    )

    r4_results.append({
        "example": i,
        "test_index": eval_indices[i - 1],
        "dialogue": example["dialogue"],
        "reference_summary": example["summary"],
        "fine_tuned_summary": fine_tuned_summary
    })

print("Post-fine-tuning inference completed.")

Post-fine-tuning inference completed.


### 5.1 Fine-Tuned Model Outputs

The following outputs show the summaries generated by the `r = 4` LoRA fine-tuned model.

The same test dialogues and generation settings used for the baseline are reused here.

In [40]:
for result in r4_results:

    print("=" * 80)
    print(f"FINE-TUNED EXAMPLE {result['example']}")
    print("=" * 80)

    print("\nREFERENCE SUMMARY:")
    print(result["reference_summary"])

    print("\nFINE-TUNED r=4 SUMMARY:")
    print(result["fine_tuned_summary"])

    print()

FINE-TUNED EXAMPLE 1

REFERENCE SUMMARY:
#Person1# rent a car from ABC Rent-a-car Company this morning and met an accident. #Person2# will call an ambulance and police for #Person1#.

FINE-TUNED r=4 SUMMARY:
#Person1# and #Person2# are on the way to Niagara Falls. #Person1#'s friend is seriously injured and #Person2# will call an ambulance and the police.

FINE-TUNED EXAMPLE 2

REFERENCE SUMMARY:
#Person1# and #Person2# will do a group report, so they plan to buy something to make posters. They make a shopping list before they go and find items according to signs on shelves.

FINE-TUNED r=4 SUMMARY:
#Person1# and #Person2# need to do a group report tomorrow. #Person1# needs markers, color pencils and highlighters and pencils. #Person1# also needs markers, color pencils and highlighters and paper clips.



In [41]:
with open("r4_finetuned_outputs.json", "w") as file:
    json.dump(
        r4_results,
        file,
        indent=2
    )

print("Fine-tuned r=4 outputs saved to r4_finetuned_outputs.json")

Fine-tuned r=4 outputs saved to r4_finetuned_outputs.json


In [42]:
print("Evaluation Examples Used")
print("-" * 40)

for i, index in enumerate(eval_indices, start=1):
    print(f"Example {i}: test index {index}")

Evaluation Examples Used
----------------------------------------
Example 1: test index 240
Example 2: test index 216


## 6. Output Comparison



Compare the model outputs before and after LoRA fine-tuning.

I compare the baseline FLAN-T5 summaries with the summaries generated by the `r = 4` fine-tuned model using the same two test dialogues.

### Approach

For each test example, I compare:

- the human-written reference summary,
- the baseline summary before fine-tuning,
- the summary after LoRA fine-tuning.

Using the same dialogues makes it easier to see what changed after training.

In [43]:
import pandas as pd

comparison_rows = []

for i in range(2):
    comparison_rows.append({
        "Example": i + 1,
        "Test Index": eval_indices[i],
        "Reference Summary": baseline_results[i]["reference_summary"],
        "Baseline Summary": baseline_results[i]["baseline_summary"],
        "LoRA r=4 Summary": r4_results[i]["fine_tuned_summary"]
    })

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df)

,Example,Test Index,Reference Summary,Baseline Summary,LoRA r=4 Summary
0,1,240,#Person1# rent a car from ABC Rent-a-car Compa...,I ran into the guardrail and got into a car.,#Person1# and #Person2# are on the way to Niag...
1,2,216,#Person1# and #Person2# will do a group report...,I need to do a group report tomorrow. I need t...,#Person1# and #Person2# need to do a group rep...


In [44]:
for i in range(2):

    print("=" * 90)
    print(f"EXAMPLE {i + 1} — TEST INDEX {eval_indices[i]}")
    print("=" * 90)

    print("\nREFERENCE SUMMARY:")
    print(baseline_results[i]["reference_summary"])

    print("\nBEFORE FINE-TUNING:")
    print(baseline_results[i]["baseline_summary"])

    print("\nAFTER FINE-TUNING — LoRA r=4:")
    print(r4_results[i]["fine_tuned_summary"])

    print()

EXAMPLE 1 — TEST INDEX 240

REFERENCE SUMMARY:
#Person1# rent a car from ABC Rent-a-car Company this morning and met an accident. #Person2# will call an ambulance and police for #Person1#.

BEFORE FINE-TUNING:
I ran into the guardrail and got into a car.

AFTER FINE-TUNING — LoRA r=4:
#Person1# and #Person2# are on the way to Niagara Falls. #Person1#'s friend is seriously injured and #Person2# will call an ambulance and the police.

EXAMPLE 2 — TEST INDEX 216

REFERENCE SUMMARY:
#Person1# and #Person2# will do a group report, so they plan to buy something to make posters. They make a shopping list before they go and find items according to signs on shelves.

BEFORE FINE-TUNING:
I need to do a group report tomorrow. I need to do a group report tomorrow. I need to go to the stationer to buy something for the posters. I need markers, color pencils, a pair of scissors, a pair of highlighter, rulers. I need markers, color pencils and highlighters. I need markers, color pencils and highlight

In [45]:
comparison_df.to_csv(
    "baseline_vs_r4_comparison.csv",
    index=False
)

print("Comparison saved to baseline_vs_r4_comparison.csv")

Comparison saved to baseline_vs_r4_comparison.csv


### Observation

After LoRA fine-tuning, the first summary became more informative and captured more of the important details from the conversation. The second summary also identified the main group-report context, although it still contained some repeated details. Overall, the fine-tuned model showed better task-specific summarization, but the outputs were not perfect.

## 7. LoRA Rank Experiment

### Question 7

Experiment with two different LoRA ranks:

- `r = 4`
- `r = 16`

I compare how increasing the LoRA rank affects the number of trainable parameters, training behavior, and generated summary quality.

### Approach

The `r = 4` experiment is already complete.

For the second experiment, I create a fresh FLAN-T5-small model and attach a LoRA adapter with `r = 16`. I keep the dataset, seed, learning rate, batch size, number of epochs, dropout, alpha, and generation settings unchanged so that the comparison is fair.

In [46]:
import gc

# The baseline outputs and r=4 outputs are already saved,
# so these model objects are no longer needed on the GPU.
del model_r4
del trainer_r4
del data_collator_r4

if "base_model" in globals():
    del base_model

if "test_batch_r4" in globals():
    del test_batch_r4

if "test_output_r4" in globals():
    del test_output_r4

gc.collect()
torch.cuda.empty_cache()

print("Previous model objects cleared from GPU memory.")

Previous model objects cleared from GPU memory.


### 7.1 LoRA Configuration for `r = 16`

For the second experiment, I increase only the LoRA rank from `4` to `16`.

The alpha, dropout, target modules, training data, and other training settings remain unchanged.

In [47]:
LORA_RANK_R16 = 16

lora_config_r16 = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_RANK_R16,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q", "v"],
    bias="none"
)

print("LoRA Configuration — r=16")
print("-" * 50)
print(f"Rank (r)       : {LORA_RANK_R16}")
print(f"Alpha          : {LORA_ALPHA}")
print(f"Dropout        : {LORA_DROPOUT}")
print('Target modules : ["q", "v"]')

LoRA Configuration — r=16
--------------------------------------------------
Rank (r)       : 16
Alpha          : 16
Dropout        : 0.05
Target modules : ["q", "v"]


### 7.2 Creating the `r = 16` Model

I load another fresh copy of FLAN-T5-small before attaching the `r = 16` adapter.

This ensures that both rank experiments start from the same original pretrained model.

In [48]:
model_r16 = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

model_r16 = get_peft_model(
    model_r16,
    lora_config_r16
)

model_r16 = model_r16.to(device)

print("Fresh LoRA r=16 model created successfully.")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Fresh LoRA r=16 model created successfully.


### 7.3 Trainable Parameters for `r = 16`

I calculate the total and trainable parameters again to observe how increasing the LoRA rank changes adapter capacity.

In [49]:
total_params_r16, trainable_params_r16, trainable_percent_r16 = count_parameters(
    model_r16
)

print("Parameter Summary — LoRA r=16")
print("-" * 50)
print(f"Total parameters     : {total_params_r16:,}")
print(f"Trainable parameters : {trainable_params_r16:,}")
print(f"Trainable percentage : {trainable_percent_r16:.4f}%")

Parameter Summary — LoRA r=16
--------------------------------------------------
Total parameters     : 77,649,280
Trainable parameters : 688,128
Trainable percentage : 0.8862%


In [50]:
print("LoRA Parameter Comparison")
print("-" * 55)

print(f"r=4  trainable parameters : {trainable_params_r4:,}")
print(f"r=16 trainable parameters : {trainable_params_r16:,}")

parameter_increase = trainable_params_r16 / trainable_params_r4

print(f"\nr=16 has {parameter_increase:.2f}x as many trainable parameters as r=4.")

LoRA Parameter Comparison
-------------------------------------------------------
r=4  trainable parameters : 172,032
r=16 trainable parameters : 688,128

r=16 has 4.00x as many trainable parameters as r=4.


In [51]:
data_collator_r16 = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_r16
)

print("Data collator created for LoRA r=16.")

Data collator created for LoRA r=16.


### 7.4 Training Setup for `r = 16`

I reuse the same training configuration from the `r = 4` experiment.

Keeping these settings unchanged helps isolate the effect of changing the LoRA rank.

In [52]:
OUTPUT_DIR_R16 = "dialogsum_lora_r16"

training_args_r16 = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR_R16,

    num_train_epochs=2,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    learning_rate=2e-4,
    weight_decay=0.01,
    max_grad_norm=1.0,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=25,

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=False,

    report_to="none",

    seed=SEED,
    data_seed=SEED
)

print("Training configuration created for LoRA r=16.")

Training configuration created for LoRA r=16.


In [53]:
trainer_r16 = Seq2SeqTrainer(
    model=model_r16,
    args=training_args_r16,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator_r16,
    processing_class=tokenizer
)

print("Trainer created successfully for LoRA r=16.")

Trainer created successfully for LoRA r=16.


### 7.5 Pre-Training Sanity Check

Before starting the r = 16 training run, I calculate the loss on a small batch to confirm that the new model and target labels are working correctly.

In [54]:
test_batch_r16 = data_collator_r16(
    [
        tokenized_train[0],
        tokenized_train[1]
    ]
)

test_batch_r16 = {
    key: value.to(device)
    for key, value in test_batch_r16.items()
}

model_r16.eval()

with torch.no_grad():
    test_output_r16 = model_r16(**test_batch_r16)

pretraining_loss_r16 = test_output_r16.loss.item()

print("Pre-Training Sanity Check — r=16")
print("-" * 50)
print(f"Batch loss: {pretraining_loss_r16:.4f}")

Pre-Training Sanity Check — r=16
--------------------------------------------------
Batch loss: 2.2771


### 7.6 Fine-Tuning the `r = 16` Adapter

The pre-training sanity check produced a valid loss, so I now fine-tune the `r = 16` LoRA adapter.

I use the same training data and training settings as the `r = 4` experiment so that the effect of changing the LoRA rank can be compared fairly.

In [55]:
import time

start_time_r16 = time.time()

train_result_r16 = trainer_r16.train()

training_time_r16 = time.time() - start_time_r16

print("\nr=16 training completed.")
print(f"Training time: {training_time_r16 / 60:.2f} minutes")

Epoch,Training Loss,Validation Loss
1,3.376159,1.446143
2,3.217955,1.419776



r=16 training completed.
Training time: 2.79 minutes


### 7.7 Training Results for r = 16

After training, I record the average training loss, validation loss, number of training steps, and training time for the `r = 16` experiment.

In [56]:
average_training_loss_r16 = train_result_r16.training_loss

eval_results_r16 = trainer_r16.evaluate()
validation_loss_r16 = eval_results_r16["eval_loss"]

print("LoRA r=16 Training Results")
print("-" * 50)
print(f"Average training loss : {average_training_loss_r16:.4f}")
print(f"Validation loss       : {validation_loss_r16:.4f}")
print(f"Global training steps : {train_result_r16.global_step}")
print(f"Training time         : {training_time_r16 / 60:.2f} minutes")

Training Loss,Validation Loss,Epoch
3.217955,1.419776,2


LoRA r=16 Training Results
--------------------------------------------------
Average training loss : 3.4987
Validation loss       : 1.4198
Global training steps : 250
Training time         : 2.79 minutes


### 7.8 Saving the r = 16 Adapter

I save the trained r = 16 LoRA adapter separately so that both trained rank experiments are preserved.



In [57]:
R16_ADAPTER_PATH = "dialogsum_lora_r16_adapter"

model_r16.save_pretrained(R16_ADAPTER_PATH)
tokenizer.save_pretrained(R16_ADAPTER_PATH)

print(f"LoRA r=16 adapter saved to: {R16_ADAPTER_PATH}")

LoRA r=16 adapter saved to: dialogsum_lora_r16_adapter


In [58]:
experiment_r16_info = {
    "model": MODEL_NAME,
    "rank": 16,
    "alpha": LORA_ALPHA,
    "dropout": LORA_DROPOUT,
    "target_modules": ["q", "v"],
    "total_parameters": total_params_r16,
    "trainable_parameters": trainable_params_r16,
    "trainable_percentage": trainable_percent_r16,
    "training_examples": len(train_data),
    "validation_examples": len(validation_data),
    "epochs": 2,
    "learning_rate": 2e-4,
    "batch_size": 8,
    "gradient_accumulation_steps": 2,
    "precision": "FP32",
    "pretraining_batch_loss": pretraining_loss_r16,
    "average_training_loss": average_training_loss_r16,
    "validation_loss": validation_loss_r16,
    "global_steps": train_result_r16.global_step,
    "training_time_minutes": training_time_r16 / 60
}

print("Final r=16 experiment metrics saved.")

Final r=16 experiment metrics saved.


### 7.9 Post-Fine-Tuning Inference for `r = 16`

I evaluate the trained `r = 16` model on the same two test dialogues used for the baseline and `r = 4` experiments.

The same generation settings are reused so that the outputs can be compared fairly.

In [59]:
model_r16.eval()

r16_results = []

for i, example in enumerate(eval_examples, start=1):

    r16_summary = generate_summary(
        model_r16,
        example["input_text"]
    )

    r16_results.append({
        "example": i,
        "test_index": eval_indices[i - 1],
        "dialogue": example["dialogue"],
        "reference_summary": example["summary"],
        "fine_tuned_summary": r16_summary
    })

print("Post-fine-tuning inference completed for LoRA r=16.")

Post-fine-tuning inference completed for LoRA r=16.


### 7.10 Generated Summaries for r = 16

The following outputs show the summaries generated by the trained r = 16 LoRA model.

In [60]:
for result in r16_results:

    print("=" * 80)
    print(f"LoRA r=16 — EXAMPLE {result['example']}")
    print("=" * 80)

    print("\nREFERENCE SUMMARY:")
    print(result["reference_summary"])

    print("\nLoRA r=16 SUMMARY:")
    print(result["fine_tuned_summary"])

    print()

LoRA r=16 — EXAMPLE 1

REFERENCE SUMMARY:
#Person1# rent a car from ABC Rent-a-car Company this morning and met an accident. #Person2# will call an ambulance and police for #Person1#.

LoRA r=16 SUMMARY:
#Person1# and #Person2# are on the way to Niagara Falls. They have a car accident near the border. #Person1# will call an ambulance and the police.

LoRA r=16 — EXAMPLE 2

REFERENCE SUMMARY:
#Person1# and #Person2# will do a group report, so they plan to buy something to make posters. They make a shopping list before they go and find items according to signs on shelves.

LoRA r=16 SUMMARY:
#Person1# and #Person2# need to do a group report tomorrow. #Person1# needs markers, color pencils and highlighters and pencils. #Person1# needs markers, color pencils and highlighters and pencils. #Person1# needs markers, color pencils and highlighters and pencils.



In [61]:
with open("r16_finetuned_outputs.json", "w") as file:
    json.dump(
        r16_results,
        file,
        indent=2
    )

print("Fine-tuned r=16 outputs saved to r16_finetuned_outputs.json")

Fine-tuned r=16 outputs saved to r16_finetuned_outputs.json


### 7.11 Comparing the Two LoRA Ranks

I compare the two experiments using their trainable parameter counts, training losses, validation losses, and training times.

I then compare the generated summaries to check whether the larger LoRA rank improved output quality.

In [62]:
rank_metrics_df = pd.DataFrame([
    {
        "LoRA Rank": 4,
        "Trainable Parameters": experiment_r4_info["trainable_parameters"],
        "Trainable %": experiment_r4_info["trainable_percentage"],
        "Average Training Loss": experiment_r4_info["average_training_loss"],
        "Validation Loss": experiment_r4_info["validation_loss"],
        "Training Time (min)": experiment_r4_info["training_time_minutes"]
    },
    {
        "LoRA Rank": 16,
        "Trainable Parameters": experiment_r16_info["trainable_parameters"],
        "Trainable %": experiment_r16_info["trainable_percentage"],
        "Average Training Loss": experiment_r16_info["average_training_loss"],
        "Validation Loss": experiment_r16_info["validation_loss"],
        "Training Time (min)": experiment_r16_info["training_time_minutes"]
    }
])

display(rank_metrics_df)

,LoRA Rank,Trainable Parameters,Trainable %,Average Training Loss,Validation Loss,Training Time (min)
0,4,172032,0.223032,3.48344,1.414560,2.700596
1,16,688128,0.886200,3.49871,1.419776,2.794206


In [63]:
rank_metrics_df.to_csv(
    "lora_rank_metrics_comparison.csv",
    index=False
)

print("LoRA rank metrics saved to lora_rank_metrics_comparison.csv")

LoRA rank metrics saved to lora_rank_metrics_comparison.csv


In [64]:
rank_output_rows = []

for i in range(2):
    rank_output_rows.append({
        "Example": i + 1,
        "Reference Summary": baseline_results[i]["reference_summary"],
        "LoRA r=4 Summary": r4_results[i]["fine_tuned_summary"],
        "LoRA r=16 Summary": r16_results[i]["fine_tuned_summary"]
    })

rank_output_df = pd.DataFrame(rank_output_rows)

display(rank_output_df)

,Example,Reference Summary,LoRA r=4 Summary,LoRA r=16 Summary
0,1,#Person1# rent a car from ABC Rent-a-car Compa...,#Person1# and #Person2# are on the way to Niag...,#Person1# and #Person2# are on the way to Niag...
1,2,#Person1# and #Person2# will do a group report...,#Person1# and #Person2# need to do a group rep...,#Person1# and #Person2# need to do a group rep...


In [65]:
for i in range(2):

    print("=" * 90)
    print(f"RANK COMPARISON — EXAMPLE {i + 1}")
    print("=" * 90)

    print("\nREFERENCE SUMMARY:")
    print(baseline_results[i]["reference_summary"])

    print("\nLoRA r=4:")
    print(r4_results[i]["fine_tuned_summary"])

    print("\nLoRA r=16:")
    print(r16_results[i]["fine_tuned_summary"])

    print()

RANK COMPARISON — EXAMPLE 1

REFERENCE SUMMARY:
#Person1# rent a car from ABC Rent-a-car Company this morning and met an accident. #Person2# will call an ambulance and police for #Person1#.

LoRA r=4:
#Person1# and #Person2# are on the way to Niagara Falls. #Person1#'s friend is seriously injured and #Person2# will call an ambulance and the police.

LoRA r=16:
#Person1# and #Person2# are on the way to Niagara Falls. They have a car accident near the border. #Person1# will call an ambulance and the police.

RANK COMPARISON — EXAMPLE 2

REFERENCE SUMMARY:
#Person1# and #Person2# will do a group report, so they plan to buy something to make posters. They make a shopping list before they go and find items according to signs on shelves.

LoRA r=4:
#Person1# and #Person2# need to do a group report tomorrow. #Person1# needs markers, color pencils and highlighters and pencils. #Person1# also needs markers, color pencils and highlighters and paper clips.

LoRA r=16:
#Person1# and #Person2# need

In [66]:
rank_output_df.to_csv(
    "r4_vs_r16_output_comparison.csv",
    index=False
)

print("r=4 vs r=16 output comparison saved.")

r=4 vs r=16 output comparison saved.


### 7.12 Observation

Increasing the LoRA rank from `r = 4` to `r = 16` increased the number of trainable parameters and gave the adapter more capacity.

However, the generated summaries were very similar for the tested dialogues, and the larger rank did not show a clear improvement in output quality. This shows that increasing LoRA rank does not automatically guarantee better summarization performance.

In [67]:
metrics_text = f"""# DATA 266 — LoRA Fine-Tuning Metrics

## Experiment Setup

- Model: {MODEL_NAME}
- Dataset: DialogSum
- Seed: {SEED}
- Training examples: {len(train_data)}
- Validation examples: {len(validation_data)}
- Epochs: 2
- Batch size: 8
- Gradient accumulation steps: 2
- Learning rate: 2e-4
- Precision: FP32
- Target modules: q, v

## LoRA Rank Comparison

| Metric | r = 4 | r = 16 |
|---|---:|---:|
| Total Parameters | {total_params_r4:,} | {total_params_r16:,} |
| Trainable Parameters | {trainable_params_r4:,} | {trainable_params_r16:,} |
| Trainable Percentage | {trainable_percent_r4:.4f}% | {trainable_percent_r16:.4f}% |
| Pre-training Batch Loss | {pretraining_loss_r4:.4f} | {pretraining_loss_r16:.4f} |
| Average Training Loss | {average_training_loss_r4:.4f} | {average_training_loss_r16:.4f} |
| Validation Loss | {validation_loss_r4:.4f} | {validation_loss_r16:.4f} |
| Training Time (minutes) | {training_time_r4 / 60:.2f} | {training_time_r16 / 60:.2f} |

## Output Comparison

The same two DialogSum test examples were used for the baseline, `r = 4`, and `r = 16` models.

The LoRA fine-tuned models produced more task-specific summaries than the original baseline on the tested examples. Increasing the rank from 4 to 16 increased adapter capacity, but it did not produce a clear improvement in summary quality for these examples.
"""

with open("METRICS.md", "w") as file:
    file.write(metrics_text)

print("METRICS.md created successfully.")

METRICS.md created successfully.


In [68]:
from datetime import datetime

run_log_text = f"""DATA 266 — LoRA Fine-Tuning Run Log
Generated: {datetime.now().isoformat()}

PERSONAL PARAMETERS
-------------------
SID4: {SID4}
SEED: {SEED}
SLICE: {SLICE}
HP_ID: {HP_ID}
CLS_A: {CLS_A}
CLS_B: {CLS_B}

RUNTIME
-------
PyTorch: {torch.__version__}
CUDA available: {torch.cuda.is_available()}
Device: {device}
GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}

DATASET
-------
Training examples: {len(train_data)}
Validation examples: {len(validation_data)}
Evaluation test indices: {eval_indices}

MODEL
-----
Base model: {MODEL_NAME}

LORA r=4
---------
Rank: 4
Alpha: {LORA_ALPHA}
Dropout: {LORA_DROPOUT}
Target modules: q, v
Trainable parameters: {trainable_params_r4:,}
Trainable percentage: {trainable_percent_r4:.4f}%
Pre-training batch loss: {pretraining_loss_r4:.4f}
Average training loss: {average_training_loss_r4:.4f}
Validation loss: {validation_loss_r4:.4f}
Global steps: {experiment_r4_info["global_steps"]}
Training time: {training_time_r4 / 60:.2f} minutes
Adapter: {R4_ADAPTER_PATH}

LORA r=16
----------
Rank: 16
Alpha: {LORA_ALPHA}
Dropout: {LORA_DROPOUT}
Target modules: q, v
Trainable parameters: {trainable_params_r16:,}
Trainable percentage: {trainable_percent_r16:.4f}%
Pre-training batch loss: {pretraining_loss_r16:.4f}
Average training loss: {average_training_loss_r16:.4f}
Validation loss: {validation_loss_r16:.4f}
Global steps: {experiment_r16_info["global_steps"]}
Training time: {training_time_r16 / 60:.2f} minutes
Adapter: {R16_ADAPTER_PATH}

OUTPUT FILES
------------
baseline_outputs.json
r4_finetuned_outputs.json
r16_finetuned_outputs.json
baseline_vs_r4_comparison.csv
lora_rank_metrics_comparison.csv
r4_vs_r16_output_comparison.csv
METRICS.md
RUN_LOG.txt
"""

with open("RUN_LOG.txt", "w") as file:
    file.write(run_log_text)

print("RUN_LOG.txt created successfully.")

RUN_LOG.txt created successfully.
